In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

/usr/local/lib/python3.10/dist-packages/yfinance/base.py:48: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  _empty_series = pd.Series()


In [ ]:
#pegar a ação que queremos trabalhar
acao = yf.Ticker("CXSE3.SA")
data = acao.history(period='1y')
df = data[['Close']]
df

,Close
Date,
2023-02-06 00:00:00-03:00,7.642248
2023-02-07 00:00:00-03:00,7.560561
2023-02-08 00:00:00-03:00,7.678553
2023-02-09 00:00:00-03:00,7.524256
2023-02-10 00:00:00-03:00,7.605943
...,...
2024-01-30 00:00:00-03:00,13.940000
2024-01-31 00:00:00-03:00,14.130000
2024-02-01 00:00:00-03:00,14.250000


In [ ]:
#Calcular a média movel
mm = df.rolling(window=20).mean()
mm

,Close
Date,
2023-02-06 00:00:00-03:00,NaN
2023-02-07 00:00:00-03:00,NaN
2023-02-08 00:00:00-03:00,NaN
2023-02-09 00:00:00-03:00,NaN
2023-02-10 00:00:00-03:00,NaN
...,...
2024-01-30 00:00:00-03:00,13.5650
2024-01-31 00:00:00-03:00,13.6315
2024-02-01 00:00:00-03:00,13.7030


In [ ]:
#Calculo do desvio padrão
dpm=df.rolling(window=20).std()
dpm

,Close
Date,
2023-02-06 00:00:00-03:00,NaN
2023-02-07 00:00:00-03:00,NaN
2023-02-08 00:00:00-03:00,NaN
2023-02-09 00:00:00-03:00,NaN
2023-02-10 00:00:00-03:00,NaN
...,...
2024-01-30 00:00:00-03:00,0.574864
2024-01-31 00:00:00-03:00,0.558403
2024-02-01 00:00:00-03:00,0.540283


In [ ]:
#Calculo da banda superior e inferior
sup_band = mm + 2 * dpm
inf_band = mm - 2 * dpm

In [ ]:
#Alterar nome das colunas de banda sup e banda inf
sup_band = sup_band.rename(columns = {'Close': 'superior'})
inf_band = inf_band.rename(columns = {'Close': 'inferior'})

In [ ]:
#unir as colunas
bandas_bollinger = df.join(sup_band).join(inf_band)
bandas_bollinger

,Close,superior,inferior
Date,,,
2023-02-06 00:00:00-03:00,7.642248,NaN,NaN
2023-02-07 00:00:00-03:00,7.560561,NaN,NaN
2023-02-08 00:00:00-03:00,7.678553,NaN,NaN
2023-02-09 00:00:00-03:00,7.524256,NaN,NaN
2023-02-10 00:00:00-03:00,7.605943,NaN,NaN
...,...,...,...
2024-01-30 00:00:00-03:00,13.940000,14.714728,12.415272
2024-01-31 00:00:00-03:00,14.130000,14.748305,12.514695
2024-02-01 00:00:00-03:00,14.250000,14.783567,12.622433


In [ ]:
#retirar os nulos
bandas_bollinger.dropna(inplace=True)

In [ ]:
#calculo dos pontos de compra e venda
compra = bandas_bollinger[bandas_bollinger['Close'] <= bandas_bollinger['inferior']]
venda = bandas_bollinger[bandas_bollinger['Close'] >= bandas_bollinger['superior']]

In [ ]:
#pip install ploty

In [ ]:
#Plotar no gráfico

import plotly.io as pio
import plotly.graph_objects as go

pio.templates.default = "plotly_dark"

fig = go.Figure()
fig.add_trace(go.Scatter(x=inf_band.index,
                         y=inf_band['inferior'],
                         name='Banda Inferior',
                         line_color='rgba(173,204,255,0.2)'
                        ))
fig.add_trace(go.Scatter(x=sup_band.index,
                         y=sup_band['superior'],
                         name='Banda Superior',
                         fill='tonexty',
                         fillcolor='rgba(173,204,255,0.2)',
                         line_color='rgba(173,204,255,0.2)'
                        ))
fig.add_trace(go.Scatter(x=df.index,
                         y=df['Close'],
                         name='Preco Fechamento',
                         line_color='#636EFA'
                        ))
fig.add_trace(go.Scatter(x=mm.index,
                         y=mm['Close'],
                         name='Media Movel',
                         line_color='#FECB52'
                        ))
fig.add_trace(go.Scatter(x=compra.index,
                         y=compra['Close'],
                         name='compra',
                         mode='markers',
                         marker=dict(
                             color='#00CC96',
                             size=8,
                             )
                         ))
fig.add_trace(go.Scatter(x=venda.index,
                         y=venda['Close'],
                         name='venda',
                         mode='markers',
                         marker=dict(
                             color='#EF553B',
                             size=8,
                             )
                         ))
fig.show()